### Declarative Pipeline Observity

- Pipeline's event log.
- Zero-config way (event_log)

- event_log('<pipeline-id>') works automatically:\
 DLT always records the event log for every pipeline internally\
event_log('pipeline-id') → built-in, always available, no setup. (What you're using now.)
- A published event-log table  only exists if you add the event_log block to pipeline settings. That's optional — useful when you want to grant access, join it in other queries, or query it from outside the pipeline.

In [0]:
%sql
SELECT timestamp, event_type, message
FROM event_log('b04e2261-f03e-4b9c-af6a-5ba6938ace32') -- pipeline id
ORDER BY timestamp DESC;

Data-quality / dropped rows (your expect_or_drop results):

In [0]:
%sql

SELECT timestamp,
  details:flow_progress.data_quality.expectations
FROM event_log('b04e2261-f03e-4b9c-af6a-5ba6938ace32')
WHERE event_type = 'flow_progress'
  AND details:flow_progress.data_quality IS NOT NULL
ORDER BY timestamp DESC;

### event_log vs System tables

### event_log('<pipeline-id>') (the TVF you're using)

- Scoped to one pipeline.
- Most detailed and real-time view — every event: flow_progress, update_progress, data-quality expectations, errors, etc.
- Same underlying data the pipeline UI shows.
- No extra permissions needed beyond access to the pipeline.


### System tables (system.*)

- Account-wide, governed, historical tables maintained by Databricks across all pipelines/jobs/clusters — e.g. system.billing.usage, system.compute, system.lakeflow.*, system.access.audit.
- Designed for cross-pipeline reporting, cost, and auditing — not the fine-grained per-batch detail.
- May have a slight delay (minutes to hours) vs. the live TVF.
- Need to be enabled by an admin and require permissions.

 So:

Want deep detail on this one pipeline (data-quality drops, per-flow progress)? → use event_log('d5a332ab-...').
Want fleet-wide trends, cost, audit history across everything? → use system.* tables.

In [0]:
files = dbutils.fs.ls("/Volumes/workspace/bronze/landing_zone/stocks/")
display(len(files))

In [0]:
%sql
DESCRIBE HISTORY workspace.bronze.stocks_bronze




In [0]:
%sql
SELECT ticker, COUNT(*) 
FROM workspace.bronze.stocks_bronze
GROUP BY ticker

In [0]:
%sql
select count(*) from workspace.bronze.stocks_bronze